In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/IS110_BPerformance_Ballego/IS110_repo_Ballego/Dashboard_Ballego

/content/drive/MyDrive/IS110_BPerformance_Ballego/IS110_repo_Ballego/Dashboard_Ballego


In [ ]:
!pwd
#for the working directory

/content/drive/MyDrive/IS110_BPerformance_Ballego/IS110_repo_Ballego/Dashboard_Ballego


In [ ]:
!pip install virtualenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 28.5 MB/s eta 0:00:00


In [ ]:
!virtualenv Michael

created virtual environment CPython3.12.13.final.0-64-x86_64 in 12370ms
  creator CPython3Posix(dest=/content/drive/MyDrive/IS110_BPerformance_Ballego/IS110_repo_Ballego/Dashboard_Ballego/Michael, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.0.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator


In [ ]:
!source Michael/bin/activate

In [ ]:
%%writefile requirements.txt
pandas
numpy

Overwriting practice.txt


In [ ]:
#install the packages in the requirements.txt
!Michael/bin/pip install -r requirements.txt

In [ ]:
# requirements.txt has been created or updated
!Michael/bin/pip freeze > requirements.txt

In [ ]:
!Michael/bin/pip list

Package Version
------- -------
pip     26.0.1


In [ ]:
!cat requirements.txt

In [ ]:
ls

dashboardPHX.ipynb  Michael/  practice.txt  requirements.txt


In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import plotly.express as px
print('Done')

Done


In [ ]:
np.random.seed(42)
random.seed(42)
categories = {
    'Electronics':  ['Laptop Pro 15', 'Wireless Mouse', 'USB Hub', 'Monitor 27"'],
    'Clothing':     ['Polo Shirt', 'Denim Jeans', 'Running Shoes', 'Cap'],
    'Food & Bev':   ['Coffee Blend 500g', 'Protein Bar Box', 'Energy Drink'],
    'Home & Living': ['Desk Lamp', 'Storage Box', 'Wall Clock'],
    'Sports':       ['Yoga Mat', 'Resistance Bands', 'Jump Rope']
}
prices = {
    'Laptop Pro 15': 65000, 'Wireless Mouse': 850, 'USB Hub': 1200, 'Monitor 27"': 18000,
    'Polo Shirt': 650,  'Denim Jeans': 1800, 'Running Shoes': 3500, 'Cap': 400,
    'Coffee Blend 500g': 380, 'Protein Bar Box': 950, 'Energy Drink': 75,
    'Desk Lamp': 1200, 'Storage Box': 550, 'Wall Clock': 800,
    'Yoga Mat': 1500, 'Resistance Bands': 600, 'Jump Rope': 350
}
regions    = ['NCR', 'Luzon', 'Visayas', 'Mindanao']
payments   = ['Credit Card', 'GCash', 'Cash on Delivery', 'Bank Transfer']
customers  = [f'Customer_{i:03d}' for i in range(1, 81)]

In [ ]:
rows = []
start = datetime(2024, 1, 1)
for i in range(1, 1001):                       # 1 000 transactions
    cat     = random.choice(list(categories))
    product = random.choice(categories[cat])
    qty     = random.randint(1, 5)
    price   = prices[product]
    date    = start + timedelta(days=random.randint(0, 364))
    rows.append({
        'order_id':       f'ORD-{1000+i}',
        'order_date':     date.strftime('%Y-%m-%d'),
        'customer_name':  random.choice(customers),
        'region':         random.choice(regions),
        'category':       cat,
        'product_name':   product,
        'quantity':       qty,
        'unit_price':     price,
        'total_amount':   qty * price,
        'payment_method': random.choice(payments)
    })

df = pd.DataFrame(rows)

df.to_csv('asset/data/sales_data.csv', index=False)

In [ ]:
df.head()

,order_id,order_date,customer_name,region,category,product_name,quantity,unit_price,total_amount,payment_method
0,ORD-1001,2024-05-05,Customer_029,Luzon,Electronics,Laptop Pro 15,3,65000,195000,Credit Card
1,ORD-1002,2024-08-04,Customer_005,NCR,Sports,Yoga Mat,5,1500,7500,Credit Card
2,ORD-1003,2024-11-04,Customer_004,Luzon,Clothing,Denim Jeans,5,1800,9000,Bank Transfer
3,ORD-1004,2024-05-22,Customer_001,Luzon,Clothing,Cap,5,400,2000,Bank Transfer
4,ORD-1005,2024-04-20,Customer_044,NCR,Food & Bev,Protein Bar Box,2,950,1900,Credit Card


In [ ]:
print(f'Generated {len(df)} rows -> asset/data/sales_data.csv')

Generated 1000 rows -> asset/data/sales_data.csv


In [ ]:
# ── 1. Load CSV ───────────────────────────────────────────────
df = pd.read_csv('asset/data/sales_data.csv')

# ── 2. Parse dates ────────────────────────────────────────────
# Convert the order_date column from string to proper datetime
df['order_date'] = pd.to_datetime(df['order_date'])

# ── 3. Extract helper columns ─────────────────────────────────
df['year']        = df['order_date'].dt.year
df['month']       = df['order_date'].dt.month
df['month_name']  = df['order_date'].dt.strftime('%b %Y')  # e.g. Jan 2024
df['month_order'] = df['order_date'].dt.to_period('M')     # for correct sorting

# ── 4. Handle missing values ──────────────────────────────────
df['total_amount'] = df['total_amount'].fillna(df['quantity'] * df['unit_price'])
df.dropna(subset=['order_id', 'order_date'], inplace=True)   # drop critical nulls

# ── 5. Ensure correct data types ──────────────────────────────
df['quantity']     = df['quantity'].astype(int)
df['unit_price']   = df['unit_price'].astype(float)
df['total_amount'] = df['total_amount'].astype(float)

# ── 6. Quick sanity check (printed to terminal on startup) ────
print(f'Loaded {len(df)} rows, date range: {df.order_date.min().date()} to {df.order_date.max().date()}')

Loaded 1000 rows, date range: 2024-01-01 to 2024-12-30


In [ ]:
# ── Monthly revenue trend ─────────────────────────────────────
monthly = (
    df.groupby(['month_order', 'month_name'], as_index=False)
      .agg(revenue=('total_amount', 'sum'), orders=('order_id', 'count'))
      .sort_values('month_order')
)

# ── Revenue by category ───────────────────────────────────────
by_category = (
    df.groupby('category', as_index=False)
      .agg(revenue=('total_amount', 'sum'), orders=('order_id', 'count'))
      .sort_values('revenue', ascending=False)
)

# ── Revenue by region ─────────────────────────────────────────
by_region = (
    df.groupby('region', as_index=False)
      .agg(revenue=('total_amount', 'sum'))
)

# ── Top 10 customers ──────────────────────────────────────────
top_customers = (
    df.groupby('customer_name', as_index=False)
      .agg(revenue=('total_amount', 'sum'), orders=('order_id', 'count'))
      .sort_values('revenue', ascending=False)
      .head(10)
)

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

def make_revenue_trend(dataframe):
    """Line chart: monthly total revenue with markers."""
    monthly = (
        dataframe.groupby(['month_order', 'month_name'], as_index=False)
                 .agg(revenue=('total_amount', 'sum'))
                 .sort_values('month_order')
    )
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=monthly['month_name'],
        y=monthly['revenue'],
        mode='lines+markers',         # line with dots at each data point
        name='Monthly Revenue',
        line=dict(color='#2E75B6', width=3),
        marker=dict(size=8, color='#1F4E79'),
        fill='tozeroy',               # shade area under the line
        fillcolor='rgba(46,117,182,0.1)'
    ))
    fig.update_layout(
        title='Monthly Revenue Trend',
        xaxis_title='Month',
        yaxis_title='Revenue (PHP)',
        plot_bgcolor='white',
        paper_bgcolor='white',
        hovermode='x unified',        # show all values at hover x position
        yaxis=dict(tickformat=',.0f') # format numbers with commas
    )
    return fig

In [ ]:
fig_revenue_trend = make_revenue_trend(df)
fig_revenue_trend.show()

In [ ]:
def make_category_bar(dataframe):
    """Horizontal bar chart: revenue per product category."""
    by_cat = (
        dataframe.groupby('category', as_index=False)
                 .agg(revenue=('total_amount', 'sum'))
                 .sort_values('revenue')      # ascending so largest is on top
    )
    fig = px.bar(
        by_cat,
        x='revenue',
        y='category',
        orientation='h',              # horizontal bars
        color='revenue',
        color_continuous_scale='Blues',
        title='Revenue by Category',
        labels={'revenue': 'Revenue (PHP)', 'category': ''},
        text_auto=',.0f'              # show value labels on bars
    )
    fig.update_layout(coloraxis_showscale=False, plot_bgcolor='white')
    return fig

In [ ]:
fig_category_bar = make_category_bar(df)
fig_category_bar.show()

In [ ]:
def make_region_pie(dataframe):
    """Donut pie chart: revenue share by region."""
    by_region = (
        dataframe.groupby('region', as_index=False)
                 .agg(revenue=('total_amount', 'sum'))
    )
    fig = px.pie(
        by_region,
        names='region',
        values='revenue',
        title='Revenue by Region',
        hole=0.4,                     # 0.4 creates a donut shape
        color_discrete_sequence=px.colors.sequential.Blues_r
    )
    fig.update_traces(textposition='outside', textinfo='percent+label')
    return fig

In [ ]:
fig_region_pie = make_region_pie(df)
fig_region_pie.show()

In [ ]:
def make_top_customers(dataframe):
    """Bar chart: top 10 customers by total spend."""
    top = (
        dataframe.groupby('customer_name', as_index=False)
                 .agg(revenue=('total_amount', 'sum'))
                 .sort_values('revenue', ascending=False)
                 .head(10)
                 .sort_values('revenue')        # flip for chart orientation
    )
    fig = px.bar(
        top,
        x='revenue', y='customer_name',
        orientation='h',
        title='Top 10 Customers by Revenue',
        labels={'revenue': 'Total Spend (PHP)', 'customer_name': ''},
        color='revenue',
        color_continuous_scale='Blues',
        text_auto=',.0f'
    )
    fig.update_layout(coloraxis_showscale=False, plot_bgcolor='white')
    return fig

In [ ]:
fig_top_customers = make_top_customers(df)
fig_top_customers.show()

In [ ]:
!Michael/bin/pip install dash dash-bootstrap-components

import dash
from dash import dcc, html, dash_table, Input, Output
import dash_bootstrap_components as dbc

# Initialize the app with a Bootstrap theme for better default styling
app = dash.Dash(
    __name__,
    external_stylesheets=[dbc.themes.BOOTSTRAP],
    title='Sales Dashboard'
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 20.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 54.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23/23 [dash-bootstrap-components]


ModuleNotFoundError: No module named 'dash'